In [ ]:
import pandas as pd
import numpy as np
import pickle
import os


from google.colab import drive
pd.set_option('display.max_columns', None)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:

# 2. Define base path correctly
base_path = '/content/drive/MyDrive/UFRGS/ESWA'

# drive.mount('/content/drive')
df = pd.read_excel(base_path+'/gpt4_cot_labes_revisada_final_fp.xlsx',sheet_name='NOVO')

In [ ]:
df

,trecho,prompt,score,topic_num,Name,ref,tipo,tipo2
0,vai usar lily roupa da planet tommy joias loui...,0,0,0,0_de_moda_marcas_consumismo,1,manual,manual
1,"Mais se os manos são do bom, bota o puma disc ...",0,0,0,0_de_moda_marcas_consumismo,1,manual,manual
2,"Na garagem um Camaro, uma Hornet\nCordão de ou...",0,0,0,0_de_moda_marcas_consumismo,1,manual,manual
3,Tá de lancha e as gata no iate\nPaco Rabanne e...,0,0,0,0_de_moda_marcas_consumismo,1,manual,manual
4,"O que eles têm, nós têm em dobro\nNós têm tant...",0,0,0,0_de_moda_marcas_consumismo,1,manual,manual
...,...,...,...,...,...,...,...,...
486,Vim de uma favela onde nunca foi fácil\nTodos ...,0,0,11,0,1,revisao_fp_at_least_2,manual
487,Você me conquistou apenas com um sorriso\nVocê...,0,0,3,0,1,revisao_fp_at_least_2,manual
488,Você me conquistou\nMe fez louco de amor\nVocê...,0,0,2,0,1,revisao_fp_at_least_2,manual
489,"Vodka ou água de coco, pra mim tanto faz Eu go...",0,0,3,0,1,revisao_fp_at_least_2,manual


In [ ]:
df['trecho'].nunique()

218

{'0': {'descricao': 'Consumo e ostentação de produtos de luxo, marcas famosas; estilo de vida luxoso',
  'positivos': ['atencao olha o ritmo quer cordao de ouro importado e um carrao ela da pra nois que nois e patrao ela da pra nois que nois e patrao ela da pra nois que nois e patrao',
   'abre espaco pros cara do momento que mete joga dentro e faz voce se apaixonar de abercrombie de christian ou ed hardy de anel uma aeropostale os cara vao pirar babar',
   'É os mente blindada da quebrada que age sem emoção\nAnda de Audi, Hornet, de Juliet\nE as novinha morre do coração\nTem que ser zika da balada\nOuro e prata destaca o aparelho\nAndar de Lacoste, de Brooksfield\nE o gelo acompanha dama de vermelho'],
  'negativos': ['Atitude, disposição só tem mesmo quem é\nNa selva urbana sobrevive só com muita fé\nÉ o som do morro que hoje invade o asfalto\nO som dos carros é o tamborzão que toca alto\nTem favelado passeando de carro importado\nDe Meriva ou de Corolla ou Honda envenenado\nCria do 

In [ ]:

dic = {  0: 'Consumo e ostentação de produto de luxos, marcas famosas; estil de vida luxoso',
  1: 'relacionamentos amorosos, intimidade, amor, paixão',
  2: 'superação de obstáculos na vida; reflexão sobre a vida e suas dificuldades; desafios e sucessos',
  3: 'desejo sexual; sedução; sexualidade; atração física; expressões vulgares',
  4: 'festa; ambientes noturnos; jovens em festas',
  5: 'consumo de álcool e drogas; consumo de substâncias',
  6: 'violência; linguagem violento; linguagem vulgar e violento',
  7: 'relações familiares; relação mãe e filho; maternidade',
  9: 'mulheres; estereótipos e aparência das mulheres',
  11: 'Vida em favela; bairros e favelas; cultura e favelas brasileiras',
  12: 'arrependimento e pedidio de perdão; arrependimento religioso; arrpendimento e pedido de perdão aos pais'
}


In [ ]:
pip install -q transformers accelerate bitsandbytes sentencepiece

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [ ]:
def get_response(messages):


  input_ids = tokenizer.apply_chat_template(
      messages,
      add_generation_prompt=True,
      return_tensors="pt"
  ).to(model.device)

  terminators = [
      tokenizer.eos_token_id,
      tokenizer.convert_tokens_to_ids("<|eot_id|>")
  ]

  outputs = model.generate(
      input_ids,
      max_new_tokens=256,
      eos_token_id=terminators,
      do_sample=True,
      temperature=0.2,
      top_p=0.9,
  )
  response = outputs[0][input_ids.shape[-1]:]
  return tokenizer.decode(response, skip_special_tokens=True)

In [ ]:
cot_prompt = { "0": { "descricao": "Consumo e ostentação de produtos de luxo, marcas famosas; estilo de vida luxoso",
                     "criterios": [ "Menciona marcas famosas (Lacoste, Audi, Louis Vuitton, etc.) de forma aspiracional ou ostensiva?",
                                   "Exibe riqueza material como símbolo de status ou poder?",
                                    "O foco central da letra é o consumo ou a posse de bens de luxo?" ],
                       "instrucao": ( "Avalie o trecho verificando cada critério acima. " "Explique quais critérios são atendidos e com que intensidade. " "Conclua com uma nota de 1 a 5, onde 1 = nenhum critério atendido e 5 = todos plenamente atendidos." ) }, "1": { "descricao": "Relacionamentos amorosos, intimidade, amor, paixão", "criterios": [ "Expressa sentimentos românticos, afeto ou paixão por outra pessoa?", "Trata de dinâmicas de relacionamento (ciúme, conflito, entrega, conquista)?", "O tema central é o vínculo emocional entre pessoas, não apenas atração física?" ], "instrucao": ( "Avalie o trecho verificando cada critério acima. " "Explique quais critérios são atendidos e com que intensidade. " "Conclua com uma nota de 1 a 5, onde 1 = nenhum critério atendido e 5 = todos plenamente atendidos." ) }, "2": { "descricao": "Superação de obstáculos na vida; reflexão sobre dificuldades; desafios e sucessos", "criterios": [ "Narra ou reflete sobre dificuldades enfrentadas (pobreza, desemprego, sofrimento)?", "Há uma mensagem de superação, esperança ou conquista apesar das adversidades?", "O foco central é a trajetória de vida difícil e a resiliência, não apenas uma descrição do ambiente?" ], "instrucao": ( "Avalie o trecho verificando cada critério acima. " "Explique quais critérios são atendidos e com que intensidade. " "Conclua com uma nota de 1 a 5, onde 1 = nenhum critério atendido e 5 = todos plenamente atendidos." ) }, "3": { "descricao": "Desejo sexual; sedução; sexualidade; atração física; expressões vulgares", "criterios": [ "Descreve atos, intenções ou fantasias sexuais de forma explícita ou sugestiva?", "Usa linguagem vulgar ou direta para tratar de sexualidade ou atração física?", "O tema central é o desejo ou a conquista sexual, não o afeto ou o relacionamento?" ], "instrucao": ( "Avalie o trecho verificando cada critério acima. " "Explique quais critérios são atendidos e com que intensidade. " "Conclua com uma nota de 1 a 5, onde 1 = nenhum critério atendido e 5 = todos plenamente atendidos." ) }, "4": { "descricao": "Festa; ambientes noturnos; jovens em festas", "criterios": [ "Descreve ou referencia um ambiente de festa, balada ou evento noturno?", "Menciona comportamentos típicos de festa (dançar, beber, se exibir, paquerar)?", "O contexto social é claramente festivo ou de lazer noturno?" ], "instrucao": ( "Avalie o trecho verificando cada critério acima. " "Explique quais critérios são atendidos e com que intensidade. " "Conclua com uma nota de 1 a 5, onde 1 = nenhum critério atendido e 5 = todos plenamente atendidos." ) }, "5": { "descricao": "Consumo de álcool e drogas; consumo de substâncias", "criterios": [ "Menciona explicitamente o consumo de álcool, drogas ou outras substâncias?", "O consumo de substâncias é central na narrativa ou apenas um detalhe contextual?", "Há glorificação, normalização ou consequências do uso de substâncias?" ], "instrucao": ( "Avalie o trecho verificando cada critério acima. " "Explique quais critérios são atendidos e com que intensidade. " "Conclua com uma nota de 1 a 5, onde 1 = nenhum critério atendido e 5 = todos plenamente atendidos." ) }, "6": { "descricao": "Violência; linguagem violenta; linguagem vulgar e violenta", "criterios": [ "Descreve atos de violência física, ameaças ou agressão?", "Usa linguagem explicitamente violenta ou intimidadora?", "O tema central é o conflito violento ou a dominância pela força, não apenas tensão emocional?" ], "instrucao": ( "Avalie o trecho verificando cada critério acima. " "Explique quais critérios são atendidos e com que intensidade. " "Conclua com uma nota de 1 a 5, onde 1 = nenhum critério atendido e 5 = todos plenamente atendidos." ) }, "7": { "descricao": "Relações familiares; relação mãe e filho; maternidade", "criterios": [ "Menciona membros da família (mãe, filho, filha, pai) com carga emocional?", "Expressa afeto, gratidão, saudade ou conflito dentro do núcleo familiar?", "O vínculo familiar é o tema central, não apenas uma menção passageira?" ], "instrucao": ( "Avalie o trecho verificando cada critério acima. " "Explique quais critérios são atendidos e com que intensidade. " "Conclua com uma nota de 1 a 5, onde 1 = nenhum critério atendido e 5 = todos plenamente atendidos." ) }, "9": { "descricao": "Mulheres; estereótipos e aparência das mulheres", "criterios": [ "Descreve ou objetifica a aparência física de mulheres?", "Reproduz estereótipos de gênero (submissão, sedução, comportamento esperado)?", "O foco do trecho é a mulher como objeto de desejo ou julgamento, não como sujeito com agência?" ], "instrucao": ( "Avalie o trecho verificando cada critério acima. " "Explique quais critérios são atendidos e com que intensidade. " "Conclua com uma nota de 1 a 5, onde 1 = nenhum critério atendido e 5 = todos plenamente atendidos." ) }, "11": { "descricao": "Vida em favela; bairros e favelas; cultura e favelas brasileiras", "criterios": [ "Referencia explicitamente a favela, morro ou periferia como espaço vivido?", "Aborda aspectos da cultura, identidade ou cotidiano da favela?", "O tema central é a vida periférica, não apenas uma menção geográfica de passagem?" ], "instrucao": ( "Avalie o trecho verificando cada critério acima. " "Explique quais critérios são atendidos e com que intensidade. " "Conclua com uma nota de 1 a 5, onde 1 = nenhum critério atendido e 5 = todos plenamente atendidos." ) }, "12": { "descricao": "Arrependimento e pedido de perdão; arrependimento religioso; arrependimento aos pais", "criterios": [ "Expressa arrependimento explícito por algo feito ou deixado de fazer?", "Há um pedido de perdão direcionado a uma pessoa ou a Deus?", "O tema central é a culpa, a reconciliação ou a redenção, não apenas tristeza genérica?" ], "instrucao": ( "Avalie o trecho verificando cada critério acima. " "Explique quais critérios são atendidos e com que intensidade. " "Conclua com uma nota de 1 a 5, onde 1 = nenhum critério atendido e 5 = todos plenamente atendidos." ) } }

In [ ]:
import re
import torch

##############################################################################
# PROMPT DO SISTEMA
##############################################################################

SYSTEM_PROMPT = """
Você é um especialista em classificação temática de letras de música.

Sua tarefa é avaliar o quanto um trecho se relaciona com um tópico específico.

Analise cuidadosamente cada critério fornecido.

Para cada critério:
- diga se foi atendido;
- explique brevemente o motivo.

Ao final escreva EXATAMENTE:

NOTA: X

onde X é um número inteiro entre 1 e 5.

Escala:

1 = nenhum critério atendido
2 = poucos critérios atendidos de forma fraca
3 = alguns critérios atendidos de forma moderada
4 = critérios fortemente presentes
5 = todos os critérios fortemente presentes
"""

##############################################################################
# GERAÇÃO
##############################################################################

def get_response(messages):

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    input_ids = inputs["input_ids"].to(model.device)

    attention_mask = inputs.get("attention_mask")

    if attention_mask is not None:
        attention_mask = attention_mask.to(model.device)

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=256,
        eos_token_id=terminators,
        do_sample=False,
        temperature=0.0,
        top_p=1.0,
    )

    response = tokenizer.decode(
        outputs[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip()

##############################################################################
# CONSTRUÇÃO DO PROMPT
##############################################################################

def build_prompt(trecho, num_topico):

    topico = cot_prompt[str(num_topico)]

    descricao = topico["descricao"]

    criterios = "\n".join(
        [f"{i+1}. {c}" for i, c in enumerate(topico["criterios"])]
    )

    instrucao = topico["instrucao"]

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"""
TÓPICO ({num_topico})

Descrição:
{descricao}

CRITÉRIOS:
{criterios}

INSTRUÇÃO:
{instrucao}

TRECHO:
\"{trecho}\"
"""
        }
    ]

    return messages

##############################################################################
# EXTRAÇÃO DA NOTA
##############################################################################

def extract_score(response):

    match = re.search(
        r"NOTA\s*:\s*([1-5])",
        response,
        flags=re.IGNORECASE
    )

    if match:
        return int(match.group(1))

    numeros = re.findall(r"\b[1-5]\b", response)

    if len(numeros) > 0:
        return int(numeros[-1])

    return None

##############################################################################
# SCORING
##############################################################################

def score_trecho(trecho, num_topico):

    prompt = build_prompt(trecho, num_topico)

    response = get_response(prompt)

    score = extract_score(response)

    return {
        "trecho": trecho,
        "num_topico": num_topico,
        "descricao": cot_prompt[str(num_topico)]["descricao"],
        "score": score,
        "response": response
    }

##############################################################################
# TESTE
##############################################################################

resultado = score_trecho(
    df["trecho"].iloc[0],
    num_topico=0
)

print("SCORE:", resultado["score"])
print()
print(resultado["response"])

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


SCORE: 5

Avaliação do trecho:

1. Menciona marcas famosas (Lacoste, Audi, Louis Vuitton, etc.) de forma aspiracional ou ostensiva?
Sim, atendido. O trecho menciona marcas famosas como Louis Vuitton e Paco Rabanne, o que sugere uma ostentação de status ou riqueza.

Intensidade: 4

2. Exibe riqueza material como símbolo de status ou poder?
Sim, atendido. O trecho apresenta uma lista de produtos de luxo, o que sugere que a riqueza material é vista como um símbolo de status ou poder.

Intensidade: 4

3. O foco central da letra é o consumo ou a posse de bens de luxo?
Sim, atendido. O trecho se concentra na descrição de produtos de luxo e marcas famosas, o que sugere que o consumo ou a posse de bens de luxo é o foco central da letra.

Intensidade: 5

NOTA: 5


In [ ]:

with open(base_path+'/chain_of_thought_2026.pkl', 'rb') as file:
            all_results = pickle.load(file)
# all_results = []

processed_pairs = set(
    (r['trecho'], r['num_topico']) for r in all_results
)

len(processed_pairs)

1571

In [ ]:
all_results

[{'trecho': 'vai usar lily roupa da planet tommy joias louis vui vip e o paco rabanne',
  'num_topico': '0',
  'descricao': 'Consumo e ostentação de produtos de luxo, marcas famosas; estilo de vida luxoso',
  'score': 4,
  'response': 'Avaliação do trecho:\n\n1. Menciona marcas famosas (Lacoste, Audi, Louis Vuitton, etc.) de forma aspiracional ou ostensiva?\nSim, atendido. O trecho menciona marcas famosas como Louis Vuitton e Paco Rabanne, o que sugere uma ostentação de status ou riqueza.\n\nIntensidade: 4\n\n2. Exibe riqueza material como símbolo de status ou poder?\nSim, atendido. O trecho apresenta uma lista de produtos de luxo, o que sugere que a riqueza material é vista como um símbolo de status ou poder.\n\nIntensidade: 4\n\n3. O foco central da letra é o consumo ou a posse de bens de luxo?\nSim, atendido. O trecho se concentra na descrição de produtos de luxo e marcas famosas, o que sugere que o consumo ou a posse de bens de luxo é o foco central da letra.\n\nIntensidade: 5\n\nN

In [ ]:

with open(base_path+'/chain_of_thought_2026.pkl', 'rb') as file:
            all_results = pickle.load(file)
# all_results = []

processed_pairs = set(
    (r['trecho'], r['num_topico']) for r in all_results
)

for trecho in df['trecho'].unique():
    for num_topico in cot_prompt.keys():
        # pula se já foi processado
        if (trecho, num_topico) in processed_pairs:
            print(f"Skipping (trecho, num_topico): ({trecho}, {num_topico})")
            continue

        response = score_trecho(trecho, num_topico=num_topico)
        print(response)

      #   result = {}
      #   result['trecho'] = trecho
      #   result['num_topico'] = num_topico
      #  # result['mean_score'] = aux_mean
      #   result['descricao'] = descricao
      #  # result['model'] = 'model'
      #   result['RESPONSE'] = response

        all_results.append(response)

        # Save the list to a file
        with open(base_path+'/chain_of_thought_2026.pkl', 'wb') as file:
            pickle.dump(all_results, file)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Skipping (trecho, num_topico): (vai usar lily roupa da planet tommy joias louis vui vip e o paco rabanne, 0)
Skipping (trecho, num_topico): (vai usar lily roupa da planet tommy joias louis vui vip e o paco rabanne, 1)
Skipping (trecho, num_topico): (vai usar lily roupa da planet tommy joias louis vui vip e o paco rabanne, 2)
Skipping (trecho, num_topico): (vai usar lily roupa da planet tommy joias louis vui vip e o paco rabanne, 3)
Skipping (trecho, num_topico): (vai usar lily roupa da planet tommy joias louis vui vip e o paco rabanne, 4)
Skipping (trecho, num_topico): (vai usar lily roupa da planet tommy joias louis vui vip e o paco rabanne, 5)
Skipping (trecho, num_topico): (vai usar lily roupa da planet tommy joias louis vui vip e o paco rabanne, 6)
Skipping (trecho, num_topico): (vai usar lily roupa da planet tommy joias louis vui vip e o paco rabanne, 7)
Skipping (trecho, num_topico): (vai usar lily roupa da planet tommy joias louis vui vip e o paco rabanne, 9)
Skipping (trecho, n

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


{'trecho': 'É tristeza e alegria ao mesmo tempo\nEssa angústia aqui dentro eu vou vivendo\nTem gente que fala mal dos morros e das favelas\nPorque não vem passar um dia em uma delas\nApesar desse problema no lugar,\nTenho certeza que um dia vai mudar', 'num_topico': '11', 'descricao': 'Vida em favela; bairros e favelas; cultura e favelas brasileiras', 'score': 3, 'response': 'Avaliação do trecho:\n\n1. Referencia explicitamente a favela, morro ou periferia como espaço vivido?\nSim, atendido. O trecho menciona "morros e favelas" explicitamente.\n\nMotivo: A menção a "morros e favelas" é clara e direta, indicando que o espaço vivido é a favela.\n\n2. Aborda aspectos da cultura, identidade ou cotidiano da favela?\nSim, atendido. O trecho aborda a percepção negativa que algumas pessoas têm sobre as favelas e a experiência de viver nesse espaço.\n\nMotivo: O trecho não apenas menciona a favela, mas também aborda a percepção e a experiência de viver nesse espaço, o que é um aspecto da cultur

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


...

{'trecho': 'Um garoto doce e humilde que não desejava o mal à ninguém\nInfelizmente os sonhos foram pro alto, levou dois tiros e morreu num assalto\nSe não reagisse seria diferente, o sonho dele era estar aqui presente com a gente\nAo lado do trono de deus descansa uma pessoa rara\nJustiça seja feita, aqui se faz, aqui se paga', 'num_topico': '7', 'descricao': 'Relações familiares; relação mãe e filho; maternidade', 'score': 2, 'response': 'Avaliação do trecho:\n\n1. Menciona membros da família (mãe, filho, filha, pai) com carga emocional?\nNão atendido. O trecho não menciona membros da família, apenas uma pessoa que morreu num assalto.\n\n2. Expressa afeto, gratidão, saudade ou conflito dentro do núcleo familiar?\nAtendido, mas de forma fraca. O trecho expressa saudade e justiça para a pessoa que morreu, mas não há um vínculo familiar explícito.\n\n3. O vínculo familiar é o tema central, não apenas uma menção passageira?\nNão atendido. O trecho não tem um vínculo familiar como tema ce

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


{'trecho': 'Um garoto doce e humilde que não desejava o mal à ninguém\nInfelizmente os sonhos foram pro alto, levou dois tiros e morreu num assalto\nSe não reagisse seria diferente, o sonho dele era estar aqui presente com a gente\nAo lado do trono de deus descansa uma pessoa rara\nJustiça seja feita, aqui se faz, aqui se paga', 'num_topico': '9', 'descricao': 'Mulheres; estereótipos e aparência das mulheres', 'score': 1, 'response': 'Avaliação do trecho:\n\n1. Descreve ou objetifica a aparência física de mulheres? - Não atendido\nO trecho não menciona a aparência física de mulheres em absoluto.\n\n2. Reproduz estereótipos de gênero (submissão, sedução, comportamento esperado)? - Não atendido\nO trecho não apresenta estereótipos de gênero, não há menção a submissão, sedução ou comportamento esperado.\n\n3. O foco do trecho é a mulher como objeto de desejo ou julgamento, não como sujeito com agência? - Não atendido\nO trecho não se refere a mulheres em absoluto, o foco é em um homem que

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


{'trecho': 'Um garoto doce e humilde que não desejava o mal à ninguém\nInfelizmente os sonhos foram pro alto, levou dois tiros e morreu num assalto\nSe não reagisse seria diferente, o sonho dele era estar aqui presente com a gente\nAo lado do trono de deus descansa uma pessoa rara\nJustiça seja feita, aqui se faz, aqui se paga', 'num_topico': '11', 'descricao': 'Vida em favela; bairros e favelas; cultura e favelas brasileiras', 'score': 3, 'response': 'Avaliação do trecho:\n\n1. Referencia explicitamente a favela, morro ou periferia como espaço vivido?\nAtendido com intensidade moderada (3). O trecho não menciona explicitamente "favela", "morro" ou "periferia", mas o contexto sugere que o garoto vive em um local com alto índice de violência e pobreza, o que é comum em favelas e bairros periféricos.\n\n2. Aborda aspectos da cultura, identidade ou cotidiano da favela?\nAtendido com intensidade fraca (2). O trecho não aborda explicitamente aspectos da cultura ou cotidiano da favela, mas o

In [ ]:
cot_prompt.keys()

dict_keys(['0', '1', '2', '3', '4', '5', '6', '7', '9', '11', '12'])